## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import copy
import os
import sys
import glob

import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torchvision import datasets
from torchvision.utils import make_grid
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

# HuggingFace Hub for persistent checkpoint storage
os.system('pip install -q huggingface_hub kaggle')
from huggingface_hub import HfApi

torch.manual_seed(17)


## Colab Setup — mount Drive + download dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Persistent checkpoint directory on Drive (survives Colab disconnects)
DRIVE_SAVE_DIR = '/content/drive/MyDrive/googlenet_imagenet100_checkpoints'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print('Checkpoints will persist at:', DRIVE_SAVE_DIR)


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ambityga/imagenet100")

print("Path to dataset files:", path)

In [ ]:
import os
base_path="/kaggle/input/imagenet100"
print(os.listdir(base_path))


## Transforms — separate train/val (augmentation only on train)

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Dataset Loading

In [ ]:
train_parts = [
    f'{base_path}/train.X1',
    f'{base_path}/train.X2',
    f'{base_path}/train.X3',
    f'{base_path}/train.X4'
]

train_datasets = [datasets.ImageFolder(root=part, transform=train_transform) for part in train_parts]
val_dataset    = datasets.ImageFolder(root=f'{base_path}/val.X', transform=val_transform)

global_classes      = sorted(set(cls for ds in train_datasets for cls in ds.classes))
global_class_to_idx = {cls: i for i, cls in enumerate(global_classes)}
NUM_CLASSES         = len(global_classes)
print(f'Global class count: {NUM_CLASSES}')

def remap_dataset(ds, mapping):
    old_idx_to_class = {v: k for k, v in ds.class_to_idx.items()}
    remap = {old_idx: mapping[cls] for old_idx, cls in old_idx_to_class.items()}
    ds.samples      = [(path, remap[label]) for path, label in ds.samples]
    ds.targets      = [remap[label] for label in ds.targets]
    ds.class_to_idx = mapping
    ds.classes      = global_classes

for ds in train_datasets:
    remap_dataset(ds, global_class_to_idx)
remap_dataset(val_dataset, global_class_to_idx)

full_train_dataset = ConcatDataset(train_datasets)

print(f'Train images : {len(full_train_dataset)}')
print(f'Val images   : {len(val_dataset)}')
print(f'Sample classes (first 5): {global_classes[:5]}')


In [ ]:
print('Merging targets...')
all_targets = np.array([label for _, label in full_train_dataset.datasets[0].samples]
                       + [label for ds in full_train_dataset.datasets[1:] for _, label in ds.samples])

print('Splitting...')
train_indices, test_indices = train_test_split(
    np.arange(len(all_targets)),
    test_size=5000,
    stratify=all_targets,
    random_state=42,
    shuffle=True
)

final_train_dataset = Subset(full_train_dataset, train_indices)
final_test_dataset  = Subset(full_train_dataset, test_indices)
print(f'Train: {len(final_train_dataset)} | Test: {len(final_test_dataset)}')


In [ ]:
batch_size = 64  # 128 causes OOM spikes on long runs; 64 is safe all session

train_loader = DataLoader(final_train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,         batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(final_test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
for images, _ in train_loader:
    plt.figure(figsize=(16, 8))
    plt.axis('off')
    plt.imshow(make_grid(images[:16], nrow=8).permute(1, 2, 0).clip(0, 1))
    break

## Model Definition

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, **kwargs):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, **kwargs)
        self.bn   = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class InceptionBlock(nn.Module):
    def __init__(self, im_channels, num_1x1, num_3x3_red, num_3x3, num_5x5_red, num_5x5, num_pool_proj):
        super(InceptionBlock, self).__init__()
        self.one_by_one        = ConvBlock(im_channels, num_1x1,       kernel_size=1)
        self.tree_by_three_red = ConvBlock(im_channels, num_3x3_red,   kernel_size=1)
        self.tree_by_three     = ConvBlock(num_3x3_red, num_3x3,       kernel_size=3, padding=1)
        self.five_by_five_red  = ConvBlock(im_channels, num_5x5_red,   kernel_size=1)
        self.five_by_five      = ConvBlock(num_5x5_red, num_5x5,       kernel_size=5, padding=2)
        self.maxpool           = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.pool_proj         = ConvBlock(im_channels, num_pool_proj, kernel_size=1)

    def forward(self, x):
        x1 = self.one_by_one(x)
        x2 = self.tree_by_three(self.tree_by_three_red(x))
        x3 = self.five_by_five(self.five_by_five_red(x))
        x4 = self.pool_proj(self.maxpool(x))
        return torch.cat([x1, x2, x3, x4], dim=1)


class Auxiliary(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(Auxiliary, self).__init__()
        self.avgpool = nn.AvgPool2d(kernel_size=5, stride=3)
        self.conv1x1 = ConvBlock(in_channels, 128, kernel_size=1)
        self.fc1     = nn.Linear(2048, 1024)
        self.fc2     = nn.Linear(1024, num_classes)
        self.dropout = nn.Dropout(0.4)   # FIX: was 0.7, too aggressive
        self.relu    = nn.ReLU()

    def forward(self, x):
        x = self.avgpool(x)
        x = self.conv1x1(x)
        x = x.reshape(x.shape[0], -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


class Inception(nn.Module):
    def __init__(self, in_channels=3, use_auxiliary=True, num_classes=100):
        super(Inception, self).__init__()
        self.conv1   = ConvBlock(in_channels, 64,  kernel_size=7, stride=2, padding=3)
        self.conv2   = ConvBlock(64,          192, kernel_size=3, stride=1, padding=1)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.avgpool = nn.AvgPool2d(kernel_size=7, stride=1)
        self.dropout = nn.Dropout(0.4)
        self.linear  = nn.Linear(1024, num_classes)

        self.use_auxiliary = use_auxiliary
        if use_auxiliary:
            self.auxiliary4a = Auxiliary(512, num_classes)
            self.auxiliary4d = Auxiliary(528, num_classes)

        self.inception3a = InceptionBlock(192, 64,  96,  128, 16, 32,  32)
        self.inception3b = InceptionBlock(256, 128, 128, 192, 32, 96,  64)
        self.inception4a = InceptionBlock(480, 192, 96,  208, 16, 48,  64)
        self.inception4b = InceptionBlock(512, 160, 112, 224, 24, 64,  64)
        self.inception4c = InceptionBlock(512, 128, 128, 256, 24, 64,  64)
        self.inception4d = InceptionBlock(512, 112, 144, 288, 32, 64,  64)
        self.inception4e = InceptionBlock(528, 256, 160, 320, 32, 128, 128)
        self.inception5a = InceptionBlock(832, 256, 160, 320, 32, 128, 128)
        self.inception5b = InceptionBlock(832, 384, 192, 384, 48, 128, 128)

    def forward(self, x):
        y = z = None
        x = self.maxpool(self.conv1(x))
        x = self.maxpool(self.conv2(x))
        x = self.maxpool(self.inception3b(self.inception3a(x)))

        x = self.inception4a(x)
        if self.training and self.use_auxiliary:
            y = self.auxiliary4a(x)

        x = self.inception4d(self.inception4c(self.inception4b(x)))
        if self.training and self.use_auxiliary:
            z = self.auxiliary4d(x)

        x = self.maxpool(self.inception4e(x))
        x = self.avgpool(self.inception5b(self.inception5a(x)))
        x = self.linear(self.dropout(x.reshape(x.shape[0], -1)))
        return x, y, z

## Device + Model Init

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

model = Inception(num_classes=NUM_CLASSES)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

model.to(device)

## Training Config

In [ ]:
EPOCHS    = 100
SAVE_DIR  = DRIVE_SAVE_DIR
os.makedirs(SAVE_DIR, exist_ok=True)  # already created above, harmless

# ── Hugging Face config ───────────────────────────────────────────────────────
# 1. Go to https://huggingface.co/settings/tokens → New token (write access)
# 2. Create a model repo at https://huggingface.co/new
# 3. Fill these in:
HF_TOKEN   = 'YOUR_HF_TOKEN'          # paste your HF write token
HF_REPO_ID = 'YOUR_HF_ID'  # e.g. chitransh001/googlenet-imagenet100
hf_api     = HfApi()

# ── Auto-detect latest checkpoint to resume from ──────────────────────────────
existing    = sorted(glob.glob(f'{SAVE_DIR}/checkpoint_epoch_*.pth'))
AUTO_RESUME = existing[-1] if existing else None
if AUTO_RESUME:
    print(f'Will resume from: {AUTO_RESUME}')
else:
    print('No checkpoint found — starting fresh')

criterion    = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer    = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)


## Checkpoint Helpers

In [ ]:
def upload_to_hf(local_path):
    """Upload a file to HuggingFace Hub. Silently skips if token not set."""
    if HF_TOKEN == '':
        return  # not configured yet
    try:
        hf_api.upload_file(
            path_or_fileobj = local_path,
            path_in_repo    = os.path.basename(local_path),
            repo_id         = HF_REPO_ID,
            repo_type       = 'model',
            token           = HF_TOKEN,
        )
        print(f'  [HF] Uploaded {os.path.basename(local_path)} -> {HF_REPO_ID}')
    except Exception as e:
        print(f'  [HF] Upload failed (non-fatal): {e}')


def save_checkpoint(epoch, model, optimizer, scheduler, best_acc, val_acc_history, path):
    state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save({
        'epoch':            epoch,
        'model_state_dict': state,
        'optim_state_dict': optimizer.state_dict(),
        'sched_state_dict': scheduler.state_dict(),
        'best_acc':         best_acc,
        'val_acc_history':  val_acc_history,
    }, path)
    print(f'  [Checkpoint] Saved -> {path}')
    sys.stdout.flush()  # force output so Kaggle doesn't think kernel is idle
    upload_to_hf(path)  # push to HF immediately after saving


def load_checkpoint(path, model, optimizer, scheduler):
    ckpt  = torch.load(path, map_location=device)
    inner = model.module if isinstance(model, nn.DataParallel) else model
    inner.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optim_state_dict'])
    scheduler.load_state_dict(ckpt['sched_state_dict'])
    print(f'  [Resume] Epoch {ckpt["epoch"]} | Best acc: {ckpt["best_acc"]:.4f}')
    return ckpt['epoch'] + 1, ckpt['best_acc'], ckpt['val_acc_history']


## Training Loop

In [ ]:
def train_model(
    model, dataloaders, criterion, optimizer, scheduler,
    num_epochs=100, use_auxiliary=True,
    save_dir=SAVE_DIR,
    checkpoint_every=1,   # every epoch — never lose more than 1 epoch
    resume_from=None,
):
    since           = time.time()
    val_acc_history = []
    best_acc        = 0.0
    best_model_wts  = copy.deepcopy(model.state_dict())
    start_epoch     = 0

    if resume_from and os.path.isfile(resume_from):
        start_epoch, best_acc, val_acc_history = load_checkpoint(
            resume_from, model, optimizer, scheduler
        )
        best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(start_epoch, num_epochs):
        print(f'\nEpoch {epoch}/{num_epochs - 1}')
        print('-' * 40)
        sys.stdout.flush()  # keep Kaggle from thinking kernel is idle

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()

            running_loss     = 0.0
            running_corrects = 0

            pbar = tqdm(dataloaders[phase], desc=f'{phase.capitalize()} {epoch+1}/{num_epochs}', leave=False)

            for inputs, labels in pbar:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    if phase == 'train':
                        if use_auxiliary:
                            outputs, aux1, aux2 = model(inputs)
                            loss = (
                                criterion(outputs, labels)
                                + 0.3 * criterion(aux1, labels)
                                + 0.3 * criterion(aux2, labels)
                            )
                        else:
                            outputs, _, _ = model(inputs)
                            loss = criterion(outputs, labels)

                        loss.backward()
                        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                        optimizer.step()
                    else:
                        outputs, _, _ = model(inputs)
                        loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)
                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                pbar.set_postfix(loss=f'{loss.item():.4f}')

                # free GPU memory every batch
                del inputs, labels, outputs
                torch.cuda.empty_cache()

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc  = running_corrects.double() / len(dataloaders[phase].dataset)
            print(f'  {phase.upper():5s} | Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}')
            sys.stdout.flush()

            if phase == 'val':
                val_acc_history.append(epoch_acc.item())

                if epoch_acc > best_acc:
                    best_acc       = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    best_path      = os.path.join(save_dir, 'best_model.pth')
                    save_checkpoint(epoch, model, optimizer, scheduler, best_acc, val_acc_history, best_path)
                    print(f'  [Best] New best val acc: {best_acc:.4f}')

        scheduler.step()
        print(f'  LR: {optimizer.param_groups[0]["lr"]:.2e}')

        # periodic checkpoint every N epochs
        if (epoch + 1) % checkpoint_every == 0:
            periodic_path = os.path.join(save_dir, f'checkpoint_epoch_{epoch+1:03d}.pth')
            save_checkpoint(epoch, model, optimizer, scheduler, best_acc, val_acc_history, periodic_path)

        print()

    elapsed = time.time() - since
    print(f'Training complete in {elapsed // 60:.0f}m {elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model, val_acc_history


## Run Training
The cell above auto-detects the latest checkpoint in `DRIVE_SAVE_DIR` (on Google Drive).
If your Colab runtime disconnects, just reconnect, re-run all cells from the top — it picks up automatically from the last saved checkpoint on Drive.
`best_model.pth` and periodic checkpoints are saved directly to your Drive, so they're safe even if the runtime is recycled.


In [ ]:
model, history = train_model(
    model,
    dataloaders      = {'train': train_loader, 'val': val_loader},
    criterion        = criterion,
    optimizer        = optimizer,
    scheduler        = lr_scheduler,
    num_epochs       = EPOCHS,
    use_auxiliary    = True,
    save_dir         = SAVE_DIR,
    checkpoint_every = 1,   # save every epoch, upload to HF immediately
    resume_from      = AUTO_RESUME,
)


## Validation Accuracy Curve

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history, marker='o', linewidth=2)
plt.title('Validation Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'val_accuracy_curve.png'))
plt.show()

## Test Set Evaluation

In [ ]:
model.eval()
correct = total = 0

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc='Testing'):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs, _, _  = model(inputs)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

print(f'Test Accuracy: {correct / total:.4f}  ({correct}/{total})')